In [ ]:
!curl -L -o ./the-babi-tasks-for-nlp-qa-system.zip https://www.kaggle.com/api/v1/datasets/download/roblexnana/the-babi-tasks-for-nlp-qa-system
!/usr/bin/unzip ./the-babi-tasks-for-nlp-qa-system.zip
!unlink ./the-babi-tasks-for-nlp-qa-system.zip

In [1]:
import sys, random, math
from collections import Counter
import numpy as np

In [2]:
f = open('./tasks_1-20_v1-2/en/qa1_single-supporting-fact_train.txt')
raw = f.readlines()
f.close()

In [6]:
sentences = list()
for line in raw[:1000]:
    sentences.append(line.lower().replace("\n", "").split(" ")[1:])

sentences[0:3]

[['mary', 'moved', 'to', 'the', 'bathroom.'],
 ['john', 'went', 'to', 'the', 'hallway.'],
 ['where', 'is', 'mary?', '\tbathroom\t1']]

In [7]:
vocab = set()
for sent in sentences:
    for word in sent:
        vocab.add(word)

vocab = list(vocab)
len(vocab)

82

In [8]:
word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

len(word2index)

82

In [9]:
def sent2indices(sent):
    indices = list()
    for word in sent:
        indices.append(word2index[word])
    return indices

In [10]:
def softmax(v):
    e_v = np.exp(v - np.max(v))
    return e_v / e_v.sum(axis=0)

In [11]:
np.random.seed(1)

embed_size = 10

# Вложение слоев
word_embeddings = (np.random.rand(len(vocab), embed_size) - 0.5) * 0.1

# Recurrent. Рекурентная матрица (первоначально единичная)
W0 = np.eye(embed_size)

# Векторное представление для пустого предложения
empty_sentence_embedding = np.zeros(embed_size)

# Decoder. Выходные веса для прогнозирования векторного предстваления предложения
W1 = (np.random.rand(embed_size, len(vocab)) - 0.5) * 0.1

# Матрица поиска выходных весов (для функции потерь)
y_hots = np.eye(len(vocab))

In [12]:
def predict(sent):
    layers = list()
    layer = {}
    layer['hidden'] = empty_sentence_embedding
    layers.append(layer)

    loss = 0

    # forward propagates
    preds = list()
    for target_i in range(len(sent)):
        layer = {}

        # tries to predict the next term
        layer['pred'] = softmax(layers[-1]['hidden'].dot(W1))

        # `sent[target_i]` gets actual word, which is a number that represent word in vocab, then we get its prediction in the proba distribution
        loss += -np.log(layer['pred'][sent[target_i]])

        # generates the next hidden state
        layer['hidden'] = layers[-1]['hidden'].dot(W0) + word_embeddings[sent[target_i]]
        layers.append(layer)

    return layers, loss

In [13]:
for iter in range(30000):
    # forward propagation
    lr = .001
    # getting sentence indices w/o 1st word
    # why it leaves the first word of each sentence? -> to use it instead of "NaN" as first layer['hidden']
    sent = sent2indices(sentences[iter % len(sentences)][1:])
    layers, loss = predict(sent)  # predicts, returns hidden layer embeddings + loss

    # backpropagation
    for layer_i in reversed(range(len(layers))):  # loop over hidden states (layers)
        layer = layers[layer_i]  # get corresponding layer
        target = sent[layer_i - 1]  # get target word

        # If not the 1st layer
        if (layer_i > 0):  # if not first layer
            layer['output_delta'] = layer['pred'] - y_hots[target]  # delta
            new_hidden_delta = layer['output_delta'].dot(W1.transpose())  # gradient

            # If last layer, don't pull from a later one, because it doesn't exist
            # seems that for each hidden layer, its hidden delta is depedent upon the last decoding operation + next layer gradient
            if (layer_i == len(layers) - 1):
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + layers[layer_i + 1]['hidden_delta'].dot(W0.transpose())
        else:  # if the first layer
            layer['hidden_delta'] = layers[layer_i + 1]['hidden_delta'].dot(W0.transpose())

    # Update weights of NaN Embedding
    empty_sentence_embedding -= layers[0]['hidden_delta'] * lr / float(len(sent))
    for layer_i, layer in enumerate(layers[1:]):
        # update decoder
        W1 -= np.outer(layers[layer_i]["hidden"], layer['output_delta']) * lr / float(len(sent))
        embed_i = sent[layer_i]
        # update embeddings
        word_embeddings[embed_i] -= layers[layer_i]['hidden_delta'] * lr / float(len(sent))
        # update encoder
        W0 -= np.outer(layers[layer_i]['hidden'], layer['hidden_delta']) * lr / float(len(sent))

    if (iter % 1000 == 0):
        print("Perplexity :" + str(np.exp(loss / len(sent))))

Perplexity :81.96137610082761
Perplexity :81.77079748525146
Perplexity :81.4755534924547
Perplexity :80.94543441473574
Perplexity :79.9045539614658
Perplexity :77.6326689255153
Perplexity :71.49469076396902
Perplexity :44.97884236722628
Perplexity :23.854822463561487
Perplexity :19.737972149312363
Perplexity :18.483103668347855
Perplexity :17.2987343667272
Perplexity :15.647122131596126
Perplexity :13.085745571369225
Perplexity :9.858549071105875
Perplexity :7.5644716141499515
Perplexity :6.411072072309123
Perplexity :5.666712646355723
Perplexity :5.16285726029949
Perplexity :4.87695310061955
Perplexity :4.685912512929502
Perplexity :4.5575717963004765
Perplexity :4.482730535129828
Perplexity :4.438246359192811
Perplexity :4.394628934267167
Perplexity :4.33783762428301
Perplexity :4.269677809398137
Perplexity :4.199643180040398
Perplexity :4.133582533005536
Perplexity :4.063786199004205


In [14]:
sent_index = 4
l, _ = predict(sent2indices(sentences[sent_index]))
print(sentences[sent_index])

['sandra', 'moved', 'to', 'the', 'garden.']


In [15]:
for i, each_layer in enumerate(l[1:-1]):
    input = sentences[sent_index][i]
    true = sentences[sent_index][i+1]
    pred = vocab[each_layer['pred'].argmax()]
    print("Prev Input:" + input + (' ' * (12 - len(input))) + "True: " + true + (' ' * (15 - len(true))) + "Pred: " + pred)

Prev Input:sandra      True: moved          Pred: is
Prev Input:moved       True: to             Pred: to
Prev Input:to          True: the            Pred: the
Prev Input:the         True: garden.        Pred: bedroom.
